# MockinJay 시나리오 테스트

## 🔬 SGLT2 억제제 논문 검색 테스트

**페르소나**: 연구자/의료진  
**의도**: RESEARCH  
**목표**: Paper DB 검색 및 최신 연구 논문 검색 검증

## 1. 라이브러리 Import

In [2]:
import requests
import json
from datetime import datetime
from pprint import pprint

## 2. 설정

In [ ]:
# API 엔드포인트
BASE_URL = "http://localhost:8000"

# 테스트 질문 - RESEARCH 의도로 Paper DB 검색
question = "SGLT2 억제제가 만성 콩팥병 환자에게 효과가 있나요? 최신 논문을 찾아주세요."
user_id = "researcher_001"

print("✅ 설정 완료")
print(f"📝 질문: {question}")
print(f"👤 사용자: {user_id}")

## 3. 서버 상태 확인

In [4]:
# Health check
try:
    response = requests.get(f"{BASE_URL}/health", timeout=5)
    health = response.json()
    print("✅ 서버 정상 동작")
    pprint(health)
except Exception as e:
    print(f"❌ 서버 연결 실패: {e}")
    print("\n💡 서버를 먼저 실행하세요:")
    print("   uvicorn backend.main:app --reload")

✅ 서버 정상 동작
{'environment': 'development',
 'service': 'MockinJay Backend',
 'status': 'healthy',
 'version': '0.1.0'}


## 4. 테스트 실행

SGLT2 억제제의 효과에 대한 최신 연구 논문을 검색합니다.

In [ ]:
print("=" * 80)
print("🔬 SGLT2 억제제 논문 검색 테스트")
print("=" * 80)

# API 요청 준비
payload = {
    "question": question,
    "user_id": user_id
}

print("\n⏳ 요청 전송 중...")
start_time = datetime.now()

try:
    response = requests.post(
        f"{BASE_URL}/api/v1/chat",
        json=payload,
        headers={"Content-Type": "application/json"},
        timeout=60
    )
    
    end_time = datetime.now()
    elapsed = (end_time - start_time).total_seconds()
    
    response.raise_for_status()
    result = response.json()
    
    print(f"\n✅ 응답 성공 (처리 시간: {elapsed:.2f}초)")
    
except requests.exceptions.RequestException as e:
    print(f"\n❌ 요청 실패: {e}")
    result = None
except Exception as e:
    print(f"\n❌ 예상치 못한 오류: {e}")
    result = None

## 5. 결과 분석

### 5.1 기본 정보

In [6]:
if result:
    print("=" * 80)
    print("📊 기본 정보")
    print("=" * 80)
    print(f"\n⏱️  처리 시간: {result.get('processing_time', 0):.2f}초")
    print(f"🎯 의도: {result['intent']} (신뢰도: {result['intent_confidence']:.2%})")
    print(f"📚 선택된 DB: {', '.join(result['selected_databases'])}")
    print(f"📄 검색된 문서 수: {result['total_documents']}개")
else:
    print("❌ 결과 없음")

📊 기본 정보

⏱️  처리 시간: 4.15초
🎯 의도: ERROR (신뢰도: 0.00%)
📚 선택된 DB: 
📄 검색된 문서 수: 0개


### 5.2 안전 검사 결과

In [7]:
if result:
    print("=" * 80)
    print("🔍 안전 검사 결과")
    print("=" * 80)
    print(f"\n응급 상황: {'❌ 아니오' if not result['is_emergency'] else '🚨 예'}")
    print(f"의료 판단 요청: {'❌ 아니오' if not result['is_medical_judgment'] else '⚠️ 예'}")
    print(f"안전성: {'✅ 안전' if result['is_safe'] else '❌ 위험'}")

🔍 안전 검사 결과

응급 상황: ❌ 아니오
의료 판단 요청: ❌ 아니오
안전성: ✅ 안전


### 5.3 검색된 논문 요약

In [ ]:
if result and result.get('summaries'):
    print("=" * 80)
    print("📚 검색된 연구 논문")
    print("=" * 80)
    
    for i, summary in enumerate(result['summaries'][:3], 1):
        print(f"\n{i}. {summary.get('title', '제목 없음')}")
        metadata = summary.get('metadata', {})
        print(f"   저자: {metadata.get('authors', 'N/A')}")
        print(f"   저널: {metadata.get('journal', 'N/A')}")
        print(f"   연도: {metadata.get('year', 'N/A')}")
        print(f"   요약: {summary.get('summary', '')[:200]}...")
        print("-" * 80)
else:
    print("❌ 검색 결과 없음")

### 5.4 최종 응답

In [9]:
if result:
    print("=" * 80)
    print("💬 최종 응답")
    print("=" * 80)
    print(f"\n{result['answer']}")

💬 최종 응답

죄송합니다. 요청 처리 중 오류가 발생했습니다.
잠시 후 다시 시도해주시거나, 문제가 지속되면 관리자에게 문의하세요.


## 6. 검증

기대하는 결과와 비교하여 테스트 통과 여부를 확인합니다.

In [ ]:
if result:
    print("=" * 80)
    print("✅ 검증 결과")
    print("=" * 80)
    
    tests = [
        ("의도 분류", result['intent'] == 'RESEARCH', f"기대: RESEARCH, 실제: {result['intent']}"),
        ("안전성 검사", result['is_safe'] == True, f"기대: 안전, 실제: {'안전' if result['is_safe'] else '위험'}"),
        ("문서 검색", result['total_documents'] > 0, f"검색된 문서: {result['total_documents']}개"),
        ("요약 생성", len(result.get('summaries', [])) > 0, f"생성된 요약: {len(result.get('summaries', []))}개"),
        ("DB 선택", 'paper_db' in result.get('selected_databases', []), f"선택된 DB: {result.get('selected_databases', [])}")
    ]
    
    all_passed = True
    for test_name, passed, message in tests:
        status = "✅ 통과" if passed else "❌ 실패"
        print(f"\n{test_name}: {status}")
        print(f"  {message}")
        if not passed:
            all_passed = False
    
    print("\n" + "=" * 80)
    if all_passed:
        print("🎉 모든 검증 통과!")
    else:
        print("❌ 일부 검증 실패")
    print("=" * 80)
else:
    print("❌ 검증 불가: 결과 없음")

## 7. 전체 응답 JSON (선택)

디버깅을 위해 전체 응답을 확인할 수 있습니다.

In [11]:
if result:
    print("전체 JSON 응답:")
    print(json.dumps(result, indent=2, ensure_ascii=False))

전체 JSON 응답:
{
  "question": "SGLT2 억제제가 만성 콩팥병 환자에게 효과가 있나요? 최신 논문을 찾아주세요.",
  "intent": "ERROR",
  "intent_confidence": 0.0,
  "answer": "죄송합니다. 요청 처리 중 오류가 발생했습니다.\n잠시 후 다시 시도해주시거나, 문제가 지속되면 관리자에게 문의하세요.",
  "summaries": [],
  "selected_databases": [],
  "total_documents": 0,
  "processing_time": 4.1542651653289795,
  "is_emergency": false,
  "is_medical_judgment": false,
  "is_safe": true,
  "timestamp": "2025-11-07T16:30:53.564759"
}
